# Simulation 2: Probing Real Transformer Activations

In dieser Simulation untersuchen wir, wie gut die Prinzipien der **Parallel Deliberation Architecture (PDA)** auf echte Transformer-Aktivierungen übertragbar sind. 

**Ziel:** 
1. Analyse der Aktivierungs-Landschaft (Effektive Dimensionalität).
2. Validierung der SVD-basierten Subspace-Zerlegung.
3. Test der parallelen Verarbeitung in Subspaces (PDA-Prototyp).
4. Anwendung der Signalmetriken (SNR, Kohärenz) auf echte neuronale Signale.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from transformer_lens import HookedTransformer

# Pfade für Helper-Dateien setzen
sys.path.append(".")
sys.path.append(os.path.join("..", "simulation-1"))
import sim2_helpers as helpers
import sim2_metrics as metrics

# Modell laden
model = HookedTransformer.from_pretrained(
    "Qwen/Qwen3-0.6B",
    device="cuda" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True,
    dtype=torch.float32
)

n_layers = model.cfg.n_layers
print(f"Modell: {model.cfg.model_name}, {n_layers} Layer, d_model={model.cfg.d_model}")

# Test-Prompts definieren
prompts_facts = [
    "The capital of France is Paris.",
    "The chemical symbol for water is H2O.",
    "Einstein is known for the theory of relativity.",
    "The sun is a star in the center of the solar system.",
    "Humans breathe oxygen to survive."
]

prompts_reasoning = [
    "If A > B and B > C, then A must be greater than C.",
    "To solve x + 5 = 10, we subtract 5 from both sides.",
    "The next number in the sequence 2, 4, 8, 16 is 32.",
    "A triangle with three equal sides is called equilateral.",
    "If it rains, the ground gets wet. It is raining, so the ground is wet."
]

prompts_creative = [
    "Once upon a time in a galaxy far, far away,",
    "The neon lights of the city reflected in the puddles,",
    "A giant clockwork dragon roared over the mountain peak,",
    "The secret of the universe was hidden in a small tea cup,",
    "Music filled the air as the stars began to dance."
]

all_prompts = prompts_facts + prompts_reasoning + prompts_creative
print(f"Setup abgeschlossen. {len(all_prompts)} Prompts.")

## Exp A: Activation Landscape

Wir analysieren das SVD-Spektrum der Aktivierungen über verschiedene Layer hinweg.

In [ ]:
# Layer dynamisch verteilen
layers_to_check = sorted(set([
    1,
    n_layers // 4,
    n_layers // 2,
    3 * n_layers // 4,
    n_layers - 2
]))
print(f"Prüfe Layer: {layers_to_check}")

# Padding-freie Aktivierungen für korrekte SVD-Analyse
acts_clean = helpers.extract_activations(model, all_prompts, layers_to_check)

results_a = {}
for l in layers_to_check:
    results_a[l] = helpers.compute_svd_spectrum(acts_clean[l])
    pr = results_a[l]['participation_ratio']
    cum_var = results_a[l]['cumulative_variance_explained']
    n90 = int(np.searchsorted(cum_var, 0.90)) + 1
    n95 = int(np.searchsorted(cum_var, 0.95)) + 1
    print(f"Layer {l:2d}: Effective Dim (PR) = {pr:.1f}, Components for 90% = {n90}, 95% = {n95}")

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for l in layers_to_check:
    ax1.plot(results_a[l]['cumulative_variance_explained'][:100], label=f"Layer {l}")
ax1.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
ax1.set_title("Cumulative Variance Explained (Top 100)")
ax1.set_xlabel("Component Index")
ax1.legend()
ax1.grid(True)

for l in layers_to_check:
    ax2.plot(results_a[l]['singular_values'][:50], label=f"Layer {l}")
ax2.set_title("Singular Value Spectrum (Top 50)")
ax2.set_xlabel("Component Index")
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

best_layer = max(results_a, key=lambda l: results_a[l]['participation_ratio'])
print(f"\nBester Layer für Exp B-E: {best_layer} (PR={results_a[best_layer]['participation_ratio']:.1f})")

# Batch-Aktivierungen für Exp B-E (braucht konsistente Shape für Hooks)
acts_batch, tokens = helpers.extract_activations_batched(model, all_prompts, [best_layer])
print(f"Batch-Aktivierungen: {acts_batch[best_layer].shape}")

## Exp B: Subspace-Zerlegung & Rekonstruktion

Wie viele Subspaces (k) benötigen wir, um die Output-Qualität zu erhalten?

In [ ]:
layer_idx = best_layer
layer_acts = acts_batch[layer_idx]

with torch.no_grad():
    original_logits = model(tokens)

# Test: Nur Top-k SVD-Komponenten behalten, Rest auf Null
# Das testet: Wie viel Info steckt in den dominanten Richtungen?
d_model = model.cfg.d_model
k_components = [5, 10, 25, 50, 100, d_model // 4, d_model // 2, d_model]

results_b = {"k": [], "kl": [], "acc": [], "var_explained": []}

# Full SVD einmal berechnen
flat = layer_acts.reshape(-1, d_model)
mean = flat.mean(dim=0)
centered = flat - mean
U, S, Vh = torch.linalg.svd(centered.float(), full_matrices=False)
V = Vh.T

for k in k_components:
    if k > d_model:
        continue
    # Rekonstruktion mit nur Top-k Komponenten
    recon_flat = (U[:, :k] @ torch.diag(S[:k]) @ V[:, :k].T) + mean
    recon = recon_flat.reshape(layer_acts.shape)

    # Varianz erklärt
    var_exp = (S[:k]**2).sum() / (S**2).sum()

    def make_hook(r):
        def hook_fn(value, hook):
            return r.to(model.cfg.device)
        return hook_fn

    hook_name = f"blocks.{layer_idx}.hook_resid_post"
    with torch.no_grad():
        recon_logits = model.run_with_hooks(
            tokens,
            fwd_hooks=[(hook_name, make_hook(recon))]
        )

    kl = helpers.compute_kl_divergence(original_logits, recon_logits)
    acc = helpers.compute_token_accuracy(original_logits, recon_logits)

    results_b["k"].append(k)
    results_b["kl"].append(kl)
    results_b["acc"].append(acc)
    results_b["var_explained"].append(var_exp.item())
    print(f"Top-{k:4d} ({var_exp:.1%} var): KL={kl:.4f}, Accuracy={acc:.4f}")

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(results_b["k"], results_b["kl"], 'r-o')
ax1.set_xlabel("Top-k Components")
ax1.set_ylabel("KL Divergence")
ax1.set_title("Output Degradation vs. Components Kept")
ax1.set_xscale('log')
ax1.grid(True)

ax2.plot(results_b["k"], results_b["acc"], 'b-s')
ax2.set_xlabel("Top-k Components")
ax2.set_ylabel("Token Accuracy")
ax2.set_title("Token Match vs. Components Kept")
ax2.set_xscale('log')
ax2.set_ylim(0, 1.05)
ax2.grid(True)
plt.tight_layout()
plt.show()

print(f"\nKritische Frage: Ab wie vielen Komponenten ist Accuracy > 90%?")

## Exp C: Semantische Analyse

Welche Subspaces reagieren auf welche Arten von Prompts?

In [ ]:
k = 4
decomp = helpers.decompose_subspaces(acts_batch[best_layer], k)

# Aktivierung pro Subspace messen (Norm der Komponenten)
subspace_energies = []
n_tokens_per_prompt = acts_batch[best_layer].shape[1]  # padded length

for sub in decomp["subspaces"]:
    # components: [batch*pos, sub_dim]
    energy = torch.norm(sub["components"], dim=-1).reshape(len(all_prompts), n_tokens_per_prompt)
    subspace_energies.append(energy.mean(dim=1).numpy())

subspace_energies = np.array(subspace_energies)  # [k, n_prompts]

# Normalisieren pro Prompt
normed = subspace_energies / subspace_energies.sum(axis=0, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

labels = [f"F{i}" for i in range(5)] + [f"R{i}" for i in range(5)] + [f"C{i}" for i in range(5)]
ylabels = [f"Sub {i} (SV {decomp['subspaces'][i]['indices'][0]}-{decomp['subspaces'][i]['indices'][1]})" for i in range(k)]

sns.heatmap(subspace_energies, annot=True, fmt=".1f", ax=ax1, xticklabels=labels, yticklabels=ylabels)
ax1.set_title(f"Raw Subspace Energy (Layer {best_layer})")

sns.heatmap(normed, annot=True, fmt=".2f", ax=ax2, xticklabels=labels, yticklabels=[f"Sub {i}" for i in range(k)])
ax2.set_title("Normalized (share per prompt)")
plt.tight_layout()
plt.show()

for i in range(k):
    f = subspace_energies[i, :5].mean()
    r = subspace_energies[i, 5:10].mean()
    c = subspace_energies[i, 10:].mean()
    print(f"Subspace {i}: Facts={f:.2f}, Reasoning={r:.2f}, Creative={c:.2f}")

## Exp D: Parallele Subspace-Verarbeitung (PDA-Test)

Wir schicken die Subspaces getrennt durch den nächsten Layer und mergen sie danach.

In [ ]:
layer_idx = best_layer
k_values_d = [2, 3, 4, 5]

# Original-Output des nächsten Layers
output_hook = f"blocks.{layer_idx+1}.hook_resid_post"
with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter=[output_hook])
    original_next_act = cache[output_hook].detach().cpu()
    del cache
    torch.cuda.empty_cache()

results_d = {"k": [], "mse": [], "cos_sim": []}

for k in k_values_d:
    print(f"\nk={k}: Parallele Verarbeitung...")
    parallel_outputs = helpers.parallel_forward(model, acts_batch[layer_idx], layer_idx, k, tokens=tokens)
    merged_output = helpers.merge_subspace_outputs(parallel_outputs)

    mse = torch.mean((merged_output - original_next_act)**2).item()
    cos_sim = torch.nn.functional.cosine_similarity(
        merged_output.reshape(-1, merged_output.shape[-1]),
        original_next_act.reshape(-1, original_next_act.shape[-1]),
        dim=1
    ).mean().item()

    results_d["k"].append(k)
    results_d["mse"].append(mse)
    results_d["cos_sim"].append(cos_sim)
    print(f"  MSE: {mse:.6f}, Cosine Sim: {cos_sim:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar([str(k) for k in results_d["k"]], results_d["mse"])
ax1.set_title("MSE (lower = better)")
ax1.set_xlabel("k subspaces")
ax2.bar([str(k) for k in results_d["k"]], results_d["cos_sim"])
ax2.set_title("Cosine Similarity (higher = better)")
ax2.set_xlabel("k subspaces")
ax2.set_ylim(0, 1)
plt.suptitle(f"PDA Parallel Forward Test (Layer {layer_idx})")
plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
best_cos = max(results_d["cos_sim"])
if best_cos > 0.9:
    print(f"POSITIV: Cosine Sim {best_cos:.4f} > 0.9 -- Subspaces getrennt verarbeitbar!")
elif best_cos > 0.7:
    print(f"GEMISCHT: Cosine Sim {best_cos:.4f} -- teilweise, Cross-Attention nötig")
else:
    print(f"NEGATIV: Cosine Sim {best_cos:.4f} < 0.7 -- Subspaces zu verflochten")

## Exp E: Signalmetriken auf echten Aktivierungen

Anwendung der SNR- und Kohärenz-Metriken.

In [ ]:
print("Berechne Signalmetriken für parallele Outputs (k=3)...")

parallel_outputs_e = helpers.parallel_forward(model, acts_batch[best_layer], best_layer, 3, tokens=tokens)

snr = metrics.compute_snr(parallel_outputs_e)
coherence = metrics.compute_phase_coherence(parallel_outputs_e)
crest = metrics.compute_crest_factor(parallel_outputs_e)

print(f"Signal-Metriken (k=3 Subspaces, Layer {best_layer}):")
print(f"  SNR: {snr:.2f} dB")
print(f"  Phase Coherence (Top 5): {np.round(coherence, 3)}")
print(f"  Crest Factor: {crest:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(range(len(coherence)), coherence)
ax1.set_title("Phase Coherence per SVD Component")
ax1.set_xlabel("Component")
ax1.set_ylabel("Coherence")

# SNR über verschiedene k
snrs = []
for k in k_values_d:
    po = helpers.parallel_forward(model, acts_batch[best_layer], best_layer, k, tokens=tokens)
    snrs.append(metrics.compute_snr(po))
ax2.bar([str(k) for k in k_values_d], snrs)
ax2.set_title("SNR vs k")
ax2.set_xlabel("k subspaces")
ax2.set_ylabel("SNR (dB)")
plt.tight_layout()
plt.show()

if snr > 0:
    print(f"SNR > 0 dB: Signal dominiert -- Subspaces sind kohärent")
else:
    print(f"SNR < 0 dB: Noise dominiert -- Subspaces divergieren")

print("=" * 60)
print("SIMULATION 2: ERGEBNIS-ZUSAMMENFASSUNG")
print("=" * 60)

print(f"\nModell: {model.cfg.model_name}, {n_layers} Layer")
print(f"Bester Layer: {best_layer} (PR={results_a[best_layer]['participation_ratio']:.1f})")

print(f"\nExp A (Activation Landscape):")
for l in layers_to_check:
    pr = results_a[l]['participation_ratio']
    print(f"  Layer {l}: PR={pr:.1f}")

print(f"\nExp B (Rekonstruktion, Layer {layer_idx}):")
for i, k in enumerate(results_b["k"]):
    print(f"  k={k}: KL={results_b['kl'][i]:.4f}, Acc={results_b['acc'][i]:.4f}")

print(f"\nExp D (Parallele Verarbeitung, Layer {layer_idx}):")
for i, k in enumerate(results_d["k"]):
    print(f"  k={k}: MSE={results_d['mse'][i]:.6f}, CosSim={results_d['cos_sim'][i]:.4f}")

print(f"\nExp E (Signalmetriken):")
print(f"  SNR={snr:.2f} dB, Crest={crest:.2f}")

print(f"\n{'='*60}")
print("ENTSCHEIDUNG:")
best_cos = max(results_d["cos_sim"])
if best_cos > 0.8:
    print("  -> POSITIV: Weiter zu Simulation 3 (Toy-Modell)")
elif best_cos > 0.5:
    print("  -> GEMISCHT: PDA braucht Cross-Attention zwischen Workern")
else:
    print("  -> NEGATIV: Subspaces zu verflochten für parallele Verarbeitung")
print(f"NAECHSTER SCHRITT: Simulation 3 (Toy-Modell from scratch)")

## Exp A2: Activation Landscape über verschiedene Modelle und Architekturen

Ist PR=1.0 ab Layer 7 ein Artefakt der Modellgröße oder ein generelles Phänomen?
Test: Qwen3-8B (4-bit) und Ministral-8B (4-bit) — verschiedene Größe UND Architektur.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc

try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

def extract_acts_hf(model_hf, tokenizer, prompts, layers):
    """Extrahiert Aktivierungen per register_forward_hook. Kein Padding."""
    all_acts = {l: [] for l in layers}

    for prompt in prompts:
        captured = {}
        hooks = []

        for l in layers:
            layer_module = model_hf.model.layers[l]
            def make_hook(layer_idx):
                def hook_fn(module, input, output):
                    # output kann tuple oder tensor sein
                    hidden = output[0] if isinstance(output, tuple) else output
                    captured[layer_idx] = hidden.detach().cpu().float()
                return hook_fn
            hooks.append(layer_module.register_forward_hook(make_hook(l)))

        inputs = tokenizer(prompt, return_tensors="pt").to(model_hf.device)
        with torch.no_grad():
            model_hf(**inputs)

        for l in layers:
            if l in captured:
                act = captured[l]
                # Shape normalisieren: [batch, seq, d] -> [seq, d]
                if act.ndim == 3:
                    act = act[0]
                elif act.ndim == 1:
                    act = act.unsqueeze(0)
                all_acts[l].append(act)

        for h in hooks:
            h.remove()

    result = {}
    for l in layers:
        if all_acts[l]:
            result[l] = torch.cat(all_acts[l], dim=0)
    return result

# Modelle zum Testen
models_to_test = [
    ("Qwen/Qwen3-8B", "Qwen3-8B"),
    ("mistralai/Ministral-8B-Instruct-2412", "Ministral-8B"),
]

all_model_results = {}
# Qwen3-0.6B Ergebnisse übernehmen
all_model_results["Qwen3-0.6B"] = {
    "layers": layers_to_check,
    "results": results_a,
    "n_layers": n_layers,
    "d_model": 1024,
}

In [ ]:
for model_id, model_label in models_to_test:
    print(f"\n{'='*60}")
    print(f"Lade {model_label} ({model_id}) in 4-bit...")
    print(f"{'='*60}")

    tokenizer_q = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model_q = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model_q.eval()

    nl = model_q.config.num_hidden_layers
    dm = model_q.config.hidden_size
    print(f"{model_label}: {nl} Layer, d_model={dm}")
    print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

    # Layer verteilt
    layers_q = sorted(set([
        1,
        nl // 4,
        nl // 2,
        3 * nl // 4,
        nl - 2
    ]))
    print(f"Prüfe Layer: {layers_q}")

    acts_q = extract_acts_hf(model_q, tokenizer_q, all_prompts, layers_q)

    results_q = {}
    for l in layers_q:
        results_q[l] = helpers.compute_svd_spectrum(acts_q[l])
        pr = results_q[l]['participation_ratio']
        cum_var = results_q[l]['cumulative_variance_explained']
        n90 = int(np.searchsorted(cum_var, 0.90)) + 1
        n95 = int(np.searchsorted(cum_var, 0.95)) + 1
        top5 = results_q[l]['singular_values'][:5]
        print(f"  Layer {l:2d}: PR={pr:.1f}, 90%={n90}, 95%={n95}, Top-5 SV: {np.round(top5, 1)}")

    all_model_results[model_label] = {
        "layers": layers_q,
        "results": results_q,
        "n_layers": nl,
        "d_model": dm,
    }

    # Aufräumen
    del model_q, tokenizer_q, acts_q
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM nach cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Vergleichsplot: PR über alle Modelle
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
positions = ["early", "1/4", "1/2", "3/4", "late"]

# PR Balkendiagramm
x = np.arange(5)
w = 0.25
for i, (label, data) in enumerate(all_model_results.items()):
    prs = [data["results"][l]["participation_ratio"] for l in data["layers"]]
    axes[0].bar(x + i*w, prs, w, label=f"{label} (d={data['d_model']})", alpha=0.8)
axes[0].set_xticks(x + w)
axes[0].set_xticklabels(positions)
axes[0].set_title("Participation Ratio per Layer Position")
axes[0].set_ylabel("PR (effektive Dimensionalität)")
axes[0].legend()
axes[0].grid(True, axis='y')

# Cumulative Variance (mittlerer Layer jedes Modells)
for label, data in all_model_results.items():
    mid_layer = data["layers"][2]  # 1/2 position
    cv = data["results"][mid_layer]["cumulative_variance_explained"][:200]
    axes[1].plot(cv, label=f"{label} L{mid_layer}")
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title("Cumulative Variance @ Middle Layer")
axes[1].set_xlabel("Component")
axes[1].legend()
axes[1].grid(True)

# SV Spectrum (mittlerer Layer, log)
for label, data in all_model_results.items():
    mid_layer = data["layers"][2]
    sv = data["results"][mid_layer]["singular_values"][:100]
    axes[2].plot(sv, label=f"{label} L{mid_layer}")
axes[2].set_title("Singular Values @ Middle Layer")
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True)
plt.tight_layout()
plt.show()

# Zusammenfassung
print("=" * 70)
print("VERGLEICH: Participation Ratio über Modelle")
print("=" * 70)
print(f"{'Modell':<20} {'d_model':>7} {'early':>8} {'1/4':>8} {'1/2':>8} {'3/4':>8} {'late':>8}")
print("-" * 70)
for label, data in all_model_results.items():
    prs = [data["results"][l]["participation_ratio"] for l in data["layers"]]
    pr_str = "".join(f"{p:8.1f}" for p in prs)
    print(f"{label:<20} {data['d_model']:>7} {pr_str}")

print("\nFrage: Ist PR=1 in tieferen Layern größenabhängig oder architekturübergreifend?")